# ABS Retail Turnover — Data Cleaning

Cleans Table 1 (national, by industry group) and Table 11 (state x industry subgroup) from the ABS Retail Trade release into long-format CSVs ready for SQL loading.

In [ ]:
import pandas as pd

def clean_abs_table(path, sheet_name="Data1", header_row=0, data_start_row=10):
    # Row 0 holds the combined header string, e.g. "Turnover ;  New South Wales ;  Food retailing ;"
    # Row 9 is the Series ID row (skipped); data starts row 10
    headers = pd.read_excel(path, sheet_name=sheet_name, header=None,
                             skiprows=header_row, nrows=1).iloc[0]

    df = pd.read_excel(path, sheet_name=sheet_name, header=None, skiprows=data_start_row)
    df.columns = ["date"] + list(headers[1:])

    df_long = df.melt(id_vars="date", var_name="category", value_name="turnover")
    df_long["date"] = pd.to_datetime(df_long["date"], errors="coerce")
    df_long["turnover"] = pd.to_numeric(df_long["turnover"], errors="coerce")
    df_long = df_long.dropna(subset=["date", "turnover"])

    parts = df_long["category"].str.split(";", expand=True)
    df_long["state"] = parts[1].str.strip()
    df_long["industry"] = parts[2].str.strip()
    df_long = df_long.drop(columns="category")

    df_long["year"] = df_long["date"].dt.year
    df_long["month"] = df_long["date"].dt.month
    return df_long[["date", "year", "month", "state", "industry", "turnover"]]


## Clean Table 1 — national turnover by industry group

In [ ]:
clean_table1 = clean_abs_table("../data/raw/850101.xlsx")
clean_table1 = clean_table1.drop(columns="state")  # always "Total (State)" in Table 1
clean_table1.to_csv("../data/clean/clean_table1.csv", index=False)
print(clean_table1.shape)
clean_table1.head()


## Clean Table 11 — state x industry subgroup turnover

In [ ]:
clean_table11 = clean_abs_table("../data/raw/8501011.xlsx")
clean_table11.to_csv("../data/clean/clean_table11.csv", index=False)
print(clean_table11.shape)
clean_table11.head()


## Quick sanity checks

In [ ]:
print("Table 1 industries:", sorted(clean_table1["industry"].unique()))
print()
print("Table 11 states:", sorted(clean_table11["state"].unique()))
print()
print("Table 11 industry subgroups:", sorted(clean_table11["industry"].unique()))
